# Data summary
Basic description and summary statistics for `trend_model_index_prices_cleaned.xlsx`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

file_path = '/Users/jimmyzheng0516/Desktop/Project Labs/LTCM_sp_2026/data/trend_model_index_prices_cleaned.xlsx'
df = pd.read_excel(file_path)
df['Dates'] = pd.to_datetime(df['Dates'])
df = df.sort_values('Dates').reset_index(drop=True)

assets = [c for c in df.columns if c != 'Dates']
df.head()

## Structure

In [ ]:
basic = pd.DataFrame({
    'item': ['rows', 'columns', 'assets', 'start_date', 'end_date', 'duplicate_dates', 'all_missing_values'],
    'value': [
        len(df),
        len(df.columns),
        len(assets),
        df['Dates'].min().date(),
        df['Dates'].max().date(),
        int(df['Dates'].duplicated().sum()),
        int(df.isna().sum().sum())
    ]
})
basic

In [ ]:
missing = df.isna().sum().rename('missing_count').to_frame()
missing['missing_pct'] = missing['missing_count'] / len(df)
missing

In [ ]:
df[assets].info()

## Price level summary

In [ ]:
price_stats = df[assets].describe().T
price_stats['median'] = df[assets].median()
price_stats['skew'] = df[assets].skew()
price_stats['kurtosis'] = df[assets].kurtosis()
price_stats = price_stats[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'median', 'skew', 'kurtosis']]
price_stats

In [ ]:
latest = pd.DataFrame({
    'latest_date': [df['Dates'].max().date()] * len(assets),
    'latest_price': df.loc[df.index[-1], assets].values
}, index=assets)
latest

## Price series

In [ ]:
price_indexed = df.set_index('Dates')[assets]
price_norm = price_indexed / price_indexed.iloc[0] * 100

plt.figure(figsize=(12, 6))
for c in assets:
    plt.plot(price_norm.index, price_norm[c], label=c, linewidth=1.2)
plt.title('Normalized price series (start = 100)')
plt.xlabel('')
plt.ylabel('Index')
plt.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

## Daily return summary

In [ ]:
ret = price_indexed.pct_change().dropna()

ret_stats = pd.DataFrame(index=assets)
ret_stats['mean_daily'] = ret.mean()
ret_stats['std_daily'] = ret.std()
ret_stats['min_daily'] = ret.min()
ret_stats['max_daily'] = ret.max()
ret_stats['skew'] = ret.skew()
ret_stats['kurtosis'] = ret.kurtosis()
ret_stats['annual_return_252'] = ret_stats['mean_daily'] * 252
ret_stats['annual_vol_252'] = ret_stats['std_daily'] * np.sqrt(252)
ret_stats['sharpe_0rf'] = ret_stats['annual_return_252'] / ret_stats['annual_vol_252']
ret_stats

In [ ]:
plt.figure(figsize=(12, 6))
for c in assets:
    plt.plot((1 + ret[c]).cumprod(), label=c, linewidth=1.2)
plt.title('Cumulative growth of $1 from daily returns')
plt.xlabel('')
plt.ylabel('Growth')
plt.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

## Correlation

In [ ]:
corr = ret.corr()
corr

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Daily return correlation')
plt.tight_layout()
plt.show()

## Rolling volatility

In [ ]:
roll_vol = ret.rolling(63).std() * np.sqrt(252)

plt.figure(figsize=(12, 6))
for c in assets:
    plt.plot(roll_vol.index, roll_vol[c], label=c, linewidth=1.1)
plt.title('63-day rolling annualized volatility')
plt.xlabel('')
plt.ylabel('Volatility')
plt.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
summary = pd.DataFrame({
    'first_obs': price_indexed.iloc[0],
    'last_obs': price_indexed.iloc[-1],
    'total_return': price_indexed.iloc[-1] / price_indexed.iloc[0] - 1,
    'best_day': ret.max(),
    'worst_day': ret.min()
})
summary